In [1]:
!pip install pyodbc

In [8]:
import pyodbc

conn = pyodbc.connect(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=SiluPC\\SQLEXPRESS;"
    "DATABASE=RetailSalesDB;"
    "Trusted_Connection=yes;"
)

print("SQL Server Connected Successfully!")

SQL Server Connected Successfully!


In [2]:
import pandas as pd
import pyodbc
import numpy as np

print("Libraries imported successfully!")

Libraries imported successfully!


In [3]:
print(pyodbc.drivers())

['SQL Server', 'SQL Server Native Client RDA 11.0', 'ODBC Driver 17 for SQL Server', 'ODBC Driver 18 for SQL Server', 'Microsoft Access Driver (*.mdb, *.accdb)', 'Microsoft Excel Driver (*.xls, *.xlsx, *.xlsm, *.xlsb)', 'Microsoft Access Text Driver (*.txt, *.csv)', 'Microsoft Access dBASE Driver (*.dbf, *.ndx, *.mdx)']


In [4]:
file_path = r"C:\Users\hp\OneDrive\Desktop\Retail_Sales_Project\Dataset\Cleaned - Retail_Sales_Cleaned.csv"

df = pd.read_csv(r"C:\Users\hp\OneDrive\Desktop\Retail_Sales_Project\Dataset\Cleaned - Retail_Sales_Cleaned.csv", encoding="latin1")

print("CSV loaded successfully!")
print(df.shape)
df.head()


CSV loaded successfully!
(9994, 21)


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,2016-06-12,2016-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


In [9]:
# Match CSV column names with SQL table names
df = df.rename(columns={
    "Row ID": "RowID",
    "Order ID": "OrderID",
    "Order Date": "OrderDate",
    "Ship Date": "ShipDate",
    "Ship Mode": "ShipMode",
    "Customer ID": "CustomerID",
    "Customer Name": "CustomerName",
    "Postal Code": "PostalCode",
    "Product ID": "ProductID",
    "Sub-Category": "SubCategory",
    "Product Name": "ProductName"
})

columns = [
    "RowID", "OrderID", "OrderDate", "ShipDate",
    "ShipMode", "CustomerID", "CustomerName",
    "Segment", "Country", "City", "State",
    "PostalCode", "Region", "ProductID",
    "Category", "SubCategory", "ProductName",
    "Sales", "Quantity", "Discount", "Profit"
]

df = df[columns]

# Convert date columns
df["OrderDate"] = pd.to_datetime(df["OrderDate"], errors="coerce")
df["ShipDate"] = pd.to_datetime(df["ShipDate"], errors="coerce")

# Prepare records for SQL Server
def convert_value(value):
    if pd.isna(value):
        return None
    if isinstance(value, pd.Timestamp):
        return value.to_pydatetime()
    if isinstance(value, np.generic):
        return value.item()
    return value

records = [
    tuple(convert_value(v) for v in row)
    for row in df.itertuples(index=False, name=None)
]

# Insert into SQL Server
cursor = conn.cursor()
cursor.fast_executemany = True

placeholders = ",".join(["?"] * len(columns))

sql = f"""
INSERT INTO StagingSales ({",".join(columns)})
VALUES ({placeholders})
"""

cursor.executemany(sql, records)
conn.commit()

print("Data uploaded successfully!")
print("Total records uploaded:", len(records))

Data uploaded successfully!
Total records uploaded: 9994
